# Notebook Librairie jobspy

Le but de ce notebook est de présenter les fonctions et méthodes de tri, dans le but de filtrer les offres d'emplois récupérées par `jobspy`.

`jobspy` est une librairie Python open source qui scrape les sites d'emplois.  
Parmi les sources disponibles on trouve : LinkedIn, ZipRecruiter, Indeed, Glassdoor, Google, Bayt, BDJobs.

In [1]:
from jobspy import scrape_jobs
import re
import json
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests


pd.set_option('display.max_columns', None)

## Fonctions de requête et filtre

In [10]:
def intervals_overlap(a_min: int, a_max: int, b_min: int, b_max: int) -> bool:
    return a_max >= b_min and b_max >= a_min

Méthodes de filtre d'emplois avancé

In [11]:
def matching_experience(text: str, min_xp: int, max_xp: int, display_info = False) -> bool:
    """
    - text : description/fiche de poste
    - min_xp : seuil minimal d'experience recherchée : entier positif (si > 0, en dessous poste trop junior)
    - max_xp : seuil maximale d'expérience recherchée : entier positif (au dessus post trop senior)

    Return true si l'intervalle [min_xp, max_xp] est compris dans les expériences requises du poste
    Exemples : 
        text = 'Niveau d'expérience minimum 3-5 ans'
        min_xp = 0
        max_xp = 2
        => FALSE

        text = 'expérience de 1 à 3 ans en data analyse'
        min_xp = 3
        max_xp = 5
        => TRUE

        text = 'expérience de 10 ans minimum'
        min_xp = 0
        max_xp = 2
        => FALSE

    """    
    true_text = text.replace("\n", " ")
    true_text = true_text.replace("\\", "")
    pattern = (
    r"(?:expérience|experience|xp).{0,40}?(\d+)\s*(?:[-àa/]\s*(\d+))?\s*[+\-]?\s*(?:ans?|années?|years?)"
    r"|"
    r"(\d+)\s*(?:[-àa/]\s*(\d+))?\s*[+\-]?\s*(?:ans?|années?|years?).{0,40}?(?:expérience|experience|xp|minimum|au moins|at least|profession|ship)"
)
    match = re.search(pattern, true_text.lower())
    if match:
        min_years = int(match.group(1) or match.group(3))
        max_years = match.group(2) or match.group(4)
        max_years = int(max_years) if max_years else min_years

        if display_info : 
            print(match, 'age mini : ', min_years, '/ age max : ', max_years)

        if min_years == max_years and min_years >= 20 :
            return True

        return intervals_overlap(min_years, max_years, min_xp, max_xp)
    
    return True

In [12]:
def is_alternance_stage(text: str) -> bool:
    """
    Return True si la description du poste contient au moins 2 occurrences de mots
    liés à l'alternance (alternan*) OU au stage (stage*, stagiaire*)
    """
    lower = text.lower()
    alternance_words = re.findall(r'\balternan\w*', lower)
    stage_words = re.findall(r'\bstag(?:e|iaire)\w*', lower)
    
    return len(alternance_words) >= 2 or len(stage_words) >= 2

In [13]:
def filter_jobs(jobs,
                exclude_altern_stage=True,
                exclude_keywords=None,
                min_xp = None,
                max_xp= None
                ): 
    
    """
    - exclude_altern_stage : exclusion des alternance et stage 
    - exclude_keywords : exclusion de mots-clefs present dans l'INTITULE du post : senior, lead, technicien,...
    - max_xp : seuil maximale d'expérience recherchée 
    """
    
    filtered = []
    excluded_log = {"altern_stage":[], "keywords": [], 'xp_desc': []}

    for job in jobs : 
        title = job.get("title", "").lower()
        description = job.get("description", "").lower() 

        if exclude_keywords :
            if any(kw in title for kw in exclude_keywords):
                excluded_log['keywords'].append(job['job_url'])
                continue

        if exclude_altern_stage and is_alternance_stage(description):
            excluded_log['altern_stage'].append(job['job_url'])
            continue


        if max_xp is not None or min_xp is not None:
            min_xp= min_xp if min_xp is not None else 0
            max_xp = max_xp if max_xp is not None else 100 

            if not matching_experience(text=description, min_xp=min_xp, max_xp=max_xp) :
                excluded_log['xp_desc'].append(job['job_url']) 
                continue


        filtered.append(job)

    return excluded_log, filtered
    

Filtrage des doublons avec calcul de similiartié (vectorisation + cosine distance)

In [25]:
def filter_similar_duplicates(
    offers: list[dict],
    threshold: float = 0.85,
    verbose: bool = False
) -> list[dict]:
    """
    Filtre les doublons quasi-identiques sur le champ "description".
    Retourne une copie de la liste de dicts filtrée.
    """
    texts = [offer["description"] for offer in offers]

    vectorizer = TfidfVectorizer(strip_accents="unicode", ngram_range=(1, 2))
    tfidf_matrix = vectorizer.fit_transform(texts)
    sim_matrix = cosine_similarity(tfidf_matrix)

    to_remove = set()
    for i in range(len(offers)):
        if i in to_remove:
            continue
        for j in range(i + 1, len(offers)):
            if j in to_remove:
                continue
            score = sim_matrix[i, j]
            if score >= threshold:
                to_remove.add(j)
                if verbose:
                    print(f"Doublon détecté (score={score:.1%}) :")
                    print(f"  [KEEP {i}] {offers[i]['job_url']}, Date :  {offers[i]['date_posted']}")
                    print(f"  [KEEP {i}] {offers[j]['job_url']}, Date :  {offers[j]['date_posted']}")
                    print()

    return [offer for i, offer in enumerate(offers) if i not in to_remove]

Filtrage par distance et temps de trajets

In [26]:
geolocator = Nominatim(user_agent="job_distance_app")

def _get_coords(city: str) -> tuple[float, float]:
    """Géocode une commune → (lat, lon)."""
    location = geolocator.geocode(f"{city}, France")
    if not location:
        raise ValueError(f"Commune introuvable : {city}")
    return (location.latitude, location.longitude)


def distance_vol_oiseau(city_a: str, city_b: str) -> float:
    """Retourne la distance en km à vol d'oiseau."""
    coords_a = _get_coords(city_a)
    coords_b = _get_coords(city_b)
    return round(geodesic(coords_a, coords_b).km, 2)

print(distance_vol_oiseau("Paris", "Issy-les-Moulineaux"))       # 392.45 km

6.35


In [27]:
def display_jobs(jobs, excluded=None):
    print("=" * 55)
    print (f'{len(jobs)} jobs trouvés : ')
    i = 0
    for j in jobs : 
         print(f"\n📌 {i+1} : {j.get('title', 'N/A')}")
         print(f"   Entreprise       : {j.get('company', 'N/A')}")
         print(f"   Lieu       : {j.get('location', 'N/A')}")
         print(f"   Date postée       : {j.get('date_posted', 'N/A')}")
         print(f"   Lien      : {j.get('job_url', 'N/A')}")
         i +=1


    if excluded :
         print("=" * 55)
         len_excluded = sum(len(v) for v in excluded.values())
         print (f'{len_excluded} jobs exclus : ')
         print(json.dumps(excluded, indent=2, ensure_ascii=False))

In [55]:
job_name = "data scientist"

jobs = scrape_jobs(
    site_name=["linkedin", "indeed"],
    search_term= job_name,
    location="Paris, France",
    results_wanted=10,
    hours_old=48,
    country_indeed='france', 
    linkedin_fetch_description=True
)


In [56]:
dict_jobs =  jobs.to_dict(orient="records") 
EXCLUSIONS = {
    'data scientist': [
        "senior", "sénior", "sr", "lead", "manager", "directeur", "responsable",
        "confirmé", "expérimenté", "expert", "head of", "chef de projet",
        "alternance", "alternant", "apprenti", "intern", "stage", "stagiaire",
        "technicien", "cyber", "cloud", "Power BI", 
        "analyst", "freelance"
    ] 
}

excluded, filtered = filter_jobs(dict_jobs, exclude_keywords=EXCLUSIONS[job_name], min_xp=0, max_xp = 2)
filtered_doublon = filter_similar_duplicates(filtered, verbose=True)
display_jobs(filtered, excluded)

4 jobs trouvés : 

📌 1 : Data Scientist
   Entreprise       : SoTalent
   Lieu       : Seine-St.-Denis, Île-de-France, France
   Date postée       : 2026-07-07
   Lien      : https://www.linkedin.com/jobs/view/4434548962

📌 2 : Data Scientist H/F
   Entreprise       : Swiss Life Asset Managers France
   Lieu       : Paris, Île-de-France, France
   Date postée       : 2026-07-07
   Lien      : https://www.linkedin.com/jobs/view/4429257126

📌 3 : Machine Learning Engineer
   Entreprise       : Modjo
   Lieu       : Paris, Île-de-France, France
   Date postée       : 2026-07-06
   Lien      : https://www.linkedin.com/jobs/view/4436253118

📌 4 : AI/ML Computational Scientist
   Entreprise       : Accenture
   Lieu       : 
   Date postée       : nan
   Lien      : https://www.linkedin.com/jobs/view/4434570236
16 jobs exclus : 
{
  "altern_stage": [],
  "keywords": [
    "https://fr.indeed.com/viewjob?jk=84709bc942f1d277",
    "https://fr.indeed.com/viewjob?jk=e4da33b4e34a9f5c",
    "https:

Test sur la fonction `requires_experience`



In [171]:
list_test = [
    "expérience de 10 ans en data analyse",
    "Vous justifiez d’une première expérience professionnelle d’au moins 2 ans",
    "Niveau d'expérience minimum 3 \- 5 ans",
    "you have 5+ years of experience ",
    "5\+ years of strong professional experience", 
    "Niveau d'expérience minimum 3 - 5 ans Compétences recherchées",
    "1-2 year of experience",
    "expérience** : **4 à 9 ans",
    "expérience de 4/5 ans",
    "Niveau : Junior (0-2 ans d'expérience)",
    "xpress, notre culture repose sur 175 ans",
    "Expérience professionnelle confirmée", # Faux positif
]

## A test xp_desc : "https://fr.indeed.com/viewjob?jk=324d36315554ef7a" (0-2 ans) seul FP pour le moment

for t in list_test : 
    print(matching_experience(text=t, min_xp=0, max_xp=0, display_info=True))

## OK

<re.Match object; span=(0, 20), match='expérience de 10 ans'> age mini :  10 / age max :  10
False
<re.Match object; span=(30, 73), match='expérience professionnelle d’au moins 2 ans'> age mini :  2 / age max :  2
False
<re.Match object; span=(9, 37), match='expérience minimum 3 - 5 ans'> age mini :  3 / age max :  5
False
<re.Match object; span=(9, 31), match='5+ years of experience'> age mini :  5 / age max :  5
False
<re.Match object; span=(0, 29), match='5+ years of strong profession'> age mini :  5 / age max :  5
False
<re.Match object; span=(9, 37), match='expérience minimum 3 - 5 ans'> age mini :  3 / age max :  5
False
<re.Match object; span=(0, 22), match='1-2 year of experience'> age mini :  1 / age max :  2
False
<re.Match object; span=(0, 26), match='expérience** : **4 à 9 ans'> age mini :  4 / age max :  9
False
<re.Match object; span=(0, 21), match='expérience de 4/5 ans'> age mini :  4 / age max :  5
False
<re.Match object; span=(17, 37), match="0-2 ans d'expérience"> ag